# TripMate AI — Multi-Agent Travel Planner

This notebook demonstrates the TripMate AI workflow: Destination, Hotel, Transport, Budget, and Itinerary agents. It uses the same Python modules as the FastAPI web application.

**Before running:** install the project requirements and add your Groq API key to the `.env` file in the project root. Do not paste or publish your API key in the notebook.

## 1. Install dependencies

Run this cell if the packages are not already installed in your notebook environment.

In [ ]:
%pip install -r requirements.txt

## 2. Load environment variables and check configuration

The `.env` file should be in the project root, alongside `main.py`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print('Groq API key configured:', bool(os.getenv('GROQ_API_KEY')))
print('Model:', os.getenv('GROQ_MODEL', 'openai/gpt-oss-120b'))

if not os.getenv('GROQ_API_KEY'):
    print('Add your key to the project .env file before running the agent cells.')

## 3. Define sample trip details

Change these values to experiment with a different trip.

In [ ]:
trip = {
    'origin': 'Hyderabad',
    'destination': 'Goa',
    'dates': '12–16 December',
    'days': 4,
    'travellers': 2,
    'budget': 25000,
    'currency': 'INR',
    'preferences': 'beaches, local food, sightseeing and relaxed travel'
}

trip

## 4. Run the Destination Agent

This agent recommends destinations or evaluates the destination entered above.

In [ ]:
from agents.destination_agent import run as destination_run

destination_notes = destination_run(trip)
print(destination_notes)

## 5. Run the Hotel, Transport, and Budget Agents

Each agent receives the trip details and selected destination.

In [ ]:
destination = trip['destination'] or 'Goa'

from agents.hotel_agent import run as hotel_run
from agents.transport_agent import run as transport_run
from agents.budget_agent import run as budget_run

hotel_notes = hotel_run(trip, destination)
transport_notes = transport_run(trip, destination)
budget_notes = budget_run(trip, destination)

print('HOTEL AGENT\n', hotel_notes)
print('\nTRANSPORT AGENT\n', transport_notes)
print('\nBUDGET AGENT\n', budget_notes)

## 6. Run the Itinerary Agent

The Itinerary Agent combines the outputs from the earlier agents into a day-by-day travel plan.

In [ ]:
from agents.itinerary_agent import run as itinerary_run

itinerary = itinerary_run(
    trip,
    destination,
    destination_notes,
    hotel_notes,
    transport_notes,
    budget_notes,
)

print(itinerary)

## 7. View all agent outputs together

This is a simple consolidated view of the multi-agent result.

In [ ]:
from IPython.display import display, Markdown

results = [
    ('Destination Agent', destination_notes),
    ('Hotel Agent', hotel_notes),
    ('Transport Agent', transport_notes),
    ('Budget Agent', budget_notes),
    ('Itinerary Agent', itinerary),
]

for agent_name, result in results:
    display(Markdown(f'## {agent_name}\n\n{result}'))

## 8. Start the FastAPI web application

Run the following command in a terminal from the project root, not inside a notebook cell if the notebook kernel is already busy:

```bash
uvicorn main:app --reload
```

Then open **http://127.0.0.1:8000** in your browser. The notebook demonstrates the agent logic; the browser interface is served by FastAPI.

## Notes and limitations

- The agents generate recommendations with an LLM; they do not book flights or hotels.
- Prices, opening times, transport schedules, and availability are not verified live.
- Confirm travel details with official providers before making bookings.
- Keep `.env` private and never commit your Groq API key to GitHub.